In [10]:
!pip install contextily

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [contextily]4 [contextily]ib]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.colors import ListedColormap

# 1. Load Data
aoi = gpd.read_file('data/boundaries/sumava_np.geojson')
zones = gpd.read_file('sumava_zones_2 1.geojson')



In [2]:

print("Unique ZONE values:", zones['ZONA'].unique())
print("\nValue counts:")
print(zones['ZONA'].value_counts())
print("\nColumn names:", zones.columns.tolist())
print("\nFirst few rows:")
print(zones[['ZONA']].head())

Unique ZONE values: <ArrowStringArray>
['A', 'B', 'C', 'D', 'I', 'II', 'III', 'IV']
Length: 8, dtype: str

Value counts:
ZONA
A      1
B      1
C      1
D      1
I      1
II     1
III    1
IV     1
Name: count, dtype: int64

Column names: ['OBJECTID', 'KOD', 'KAT', 'NAZEV', 'ZONA', 'ZMENA_G', 'ZMENA_T', 'PREKRYV', 'DBID', 'SHAPEAREA', 'SHAPELEN', 'geometry']

First few rows:
  ZONA
0    A
1    B
2    C
3    D
4    I


In [3]:
# 2. Reproject to Web Mercator (EPSG:3857) for OSM basemap
# Contextily requires this CRS to fetch tiles correctly
aoi_wgs = aoi.to_crs(epsg=4326) # Ensure original is WGS84 if needed
zones_wgs = zones.to_crs(epsg=4326)

# Filter for NP (National Park) only
zones_np = zones[zones['KAT'] == 'NP'].copy()

# Then use zones_np instead of zones for all subsequent operations
zones_wgs = zones_np.to_crs(epsg=4326)
zones_plot = zones_wgs.to_crs(epsg=3857)



In [ ]:
# 1. Filter for NP only
zones_np = zones[zones['KAT'] == 'NP'].copy()

# 2. Check if we have data after filtering
if len(zones_np) == 0:
    print("Warning: No NP zones found in the dataset")
else:
    print(f"Found {len(zones_np)} NP zones")
    
    # 3. Reproject to Web Mercator (EPSG:3857) for OSM basemap
    zones_wgs = zones_np.to_crs(epsg=4326)
    zones_plot = zones_wgs.to_crs(epsg=3857)
    
    # 4. Setup Plot
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # 5. Plot Zones with Custom Green Shades
    zone_colors = ['#004d00', '#006600', '#009900', '#00cc00']
    zone_labels = ['Zone A (Strict)', 'Zone B', 'Zone C', 'Zone D']
    
    cmap = ListedColormap(zone_colors)
    
    zones_plot.plot(
        ax=ax,
        column='ZONA',
        cmap=cmap,
        categorical=True,
        categories=['A', 'B', 'C', 'D'],
        legend=True,
        legend_kwds={
            'loc': 'lower left',
            'frameon': True,
            'fontsize': 10,
            'title_fontsize': 12
        },
        edgecolor='brown',
        linewidth=0.8,
        alpha=0.85,
        label='protection zone'
    )
    

# 5. Plot AOI Boundary (Red outline)
aoi_wgs.boundary.plot(ax=ax, color='green', linewidth=0.8, label='AOI Boundary')

# 6. Add OSM Basemap
# 'Stamen Terrain' is great for nature, 'OpenStreetMap' is standard
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11)


# 7. Final Styling
ax.set_title('Šumava National Park: Protection Zones & Study Area', 
             fontsize=16, fontweight='bold', pad=20)
ax.axis('off') # Hide axes for a cleaner map look
ax.set_facecolor('#f0f0f0') # Light gray background if basemap fails

plt.tight_layout()
plt.show()
fig.savefig('sumava_visualization.png', dpi=300)

Found 4 NP zones


/opt/anaconda3/lib/python3.11/site-packages/joblib/parallel.py:288: UserWarning: Persisting input arguments took 9.44s to run.
If this happens often in your code, it can cause performance problems 
(results will be correct in all cases). 
The reason for this is probably some large input arguments for a wrapped
 function (e.g. large strings).
THIS IS A JOBLIB ISSUE. If you can, kindly provide the joblib's team with an
 example so that they can fix the problem.
  return [func(*args, **kwargs)
/opt/anaconda3/lib/python3.11/site-packages/joblib/parallel.py:288: UserWarning: Persisting input arguments took 2.00s to run.
If this happens often in your code, it can cause performance problems 
(results will be correct in all cases). 
The reason for this is probably some large input arguments for a wrapped
 function (e.g. large strings).
THIS IS A JOBLIB ISSUE. If you can, kindly provide the joblib's team with an
 example so that they can fix the problem.
  return [func(*args, **kwargs)
/opt/ana